In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gegeorge250/grimm-brothers/pg2591.txt


In [2]:
import pandas as pd

In [21]:
data_home = "../input"
output_dir = "../working"

In [22]:
OHCO = ['story_num', 'para_num', 'sent_num', 'token_num']

## Turning Gutenburg File into CSV

In [5]:
pg_id = 25291
pg_title ="grimm"

/kaggle/input/datasets/gegeorge250/grimm-brothers/pg2591.txt

In [23]:
text_file = f"/kaggle/input/datasets/gegeorge250/grimm-brothers/pg2591.txt"
csv_file  = f"{output_dir}/{pg_title}.csv" # The file we will create


In [24]:
LINES = pd.DataFrame(open(text_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()

In [25]:
LINES.sample(20)

,line_str
line_num,
8743,playthings for his children. On the third day ...
143,only it dropped a golden feather from its tail...
285,"young man refused to do it: so the fox said, ‘..."
4488,had passed; and they warned her once more not ...
5959,
3127,nothing that she would not have given to the c...
4827,"so he went to his master, and said, ‘I have wo..."
4904,no one could hold the miser. At the second not...
4233,"the night, the old woman came creeping in, she..."


In [26]:
LINES.loc[0].line_str

"The Project Gutenberg eBook of Grimms' Fairy Tales"

In [27]:
title = LINES.loc[0].line_str.replace('The Project Gutenberg eBook of ', '')
print(title)

Grimms' Fairy Tales


In [28]:
clip_pats = [
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT",
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT"
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
line_a,line_b
                                 

(np.int64(28), np.int64(9267))

In [29]:
LINES = LINES.loc[line_a : line_b]
LINES.head(10)

,line_str
line_num,
28,
29,
30,
31,
32,Grimms’ Fairy Tales
33,
34,By Jacob Grimm and Wilhelm Grimm
35,
36,


In [30]:
story_pat = r"^[A-Z][A-Z\s,'\-]{5,}$"
story_lines = LINES.line_str.str.match(story_pat, na=False) # Used AI to help match titles

# Used AI to help make regx string

In [32]:
LINES.loc[story_lines] # Use as filter for dataframe


,line_str
line_num,
49,THE GOLDEN BIRD
50,HANS IN LUCK
51,JORINDA AND JORINDEL
52,THE TRAVELLING MUSICIANS
53,OLD SULTAN
...,...
8157,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...
8463,KING GRISLY-BEARD
8593,IRON HANS


In [33]:
LINES.loc[story_lines, 'story_num'] = [i+1 for i in range(LINES.loc[story_lines].shape[0])]
LINES.loc[story_lines]

,line_str,story_num
line_num,,
49,THE GOLDEN BIRD,1.0
50,HANS IN LUCK,2.0
51,JORINDA AND JORINDEL,3.0
52,THE TRAVELLING MUSICIANS,4.0
53,OLD SULTAN,5.0
...,...,...
8157,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...,123.0
8463,KING GRISLY-BEARD,124.0
8593,IRON HANS,125.0


In [34]:
LINES.story_num = LINES.story_num.ffill()
LINES.head(30)

,line_str,story_num
line_num,,
49,THE GOLDEN BIRD,1.0
50,HANS IN LUCK,2.0
51,JORINDA AND JORINDEL,3.0
52,THE TRAVELLING MUSICIANS,4.0
53,OLD SULTAN,5.0
54,"THE STRAW, THE COAL, AND THE BEAN",6.0
55,BRIAR ROSE,7.0
56,THE DOG AND THE SPARROW,8.0
57,THE TWELVE DANCING PRINCESSES,9.0


In [19]:
LINES

,line_str,story_num
line_num,,
49,THE GOLDEN BIRD,1.0
50,HANS IN LUCK,2.0
51,JORINDA AND JORINDEL,3.0
52,THE TRAVELLING MUSICIANS,4.0
53,OLD SULTAN,5.0
...,...,...
9263,,127.0
9264,,127.0
9265,,127.0


In [ ]:
LINES = LINES.dropna(subset=['story_num']) # Remove everything before Chapter 1
# LINES = LINES.loc[~LINES.chap_num.isna()] # Remove everything before Chapter 1 (alternate method)
LINES = LINES.loc[~story_lines] # Remove chapter heading lines; their work is done
LINES.story_num = LINES.story_num.astype('int') # Convert chap_num from float to int

In [ ]:
LINES.head(10)

In [ ]:
STORIES_old = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('story_str')
STORIES_old.head(6)

In [ ]:
STORIES=STORIES_old.copy()[5:] # Cut of extra stuff that did not get deleted with regex
STORIES['story_str'] = STORIES.story_str.str.strip()
STORIES.tail(25)